
# 04. Vocabulary Engineering and Sequence Construction

## Why words must become numbers
Neural models operate on tensors, not strings. Vocabulary maps text tokens to integer IDs.

## Definitions
- `word2idx`: token -> integer ID
- `idx2word`: integer ID -> token
- `<unk>`: out-of-vocabulary fallback
- `<pad>`: sequence padding token
- `<bos>/<eos>`: sequence boundaries


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.data import prepare_combined_corpus
from utils.tokenization import NLTKTokenizerBackend, normalize_text
from utils.vocabulary import Vocabulary
from utils.sequence_builder import build_context_target_pairs

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bundle = prepare_combined_corpus(
    project_root=PROJECT_ROOT,
    include_wikitext=True,
    wikitext_train_tokens=120_000,
    wikitext_val_tokens=20_000,
    wikitext_test_tokens=20_000,
)

tokens = NLTKTokenizerBackend().tokenize(normalize_text(bundle.train_text))


In [ ]:

vocab_sizes = {}
for min_freq in [1, 2, 5, 10]:
    vocab = Vocabulary(min_freq=min_freq, max_size=20_000)
    vocab.build([tokens])
    vocab_sizes[min_freq] = len(vocab)

pd.DataFrame(
    [{"min_freq": k, "vocab_size": v} for k, v in vocab_sizes.items()]
).sort_values("min_freq")


In [ ]:

vocab = Vocabulary(min_freq=2, max_size=20_000)
vocab.build([tokens])
ids = vocab.encode_with_special(tokens)

print("Vocab size:", len(vocab))
print("Sample word2idx:", list(vocab.word2idx.items())[:10])
print("Sample idx2word:", [(i, vocab.idx2word[i]) for i in range(10)])


In [ ]:

context_options = [3, 5, 10, 20]
counts = []
for context_len in context_options:
    pairs = build_context_target_pairs(ids, context_len=context_len)
    counts.append({"context_len": context_len, "num_pairs": len(pairs)})

counts_df = pd.DataFrame(counts)
counts_df


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(counts_df["context_len"], counts_df["num_pairs"], marker="o")
plt.title("Sequence count vs context window")
plt.xlabel("Context length")
plt.ylabel("Number of (context,target) pairs")
plt.show()


In [ ]:

# Save vocabulary for training + app.
out_path = PROJECT_ROOT / "outputs" / "vocab_notebook.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
vocab.save(out_path)
print("Saved:", out_path)
